In [1]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so dsppf_backbone.py can be imported

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 4.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
ultralytics 8.4.137 | torch 2.11.0+cu128 | torchvision 0.26.0+cu128
GPU: Tesla T4


In [2]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to digitalaccess-2(copy)-4 in yolo26:: 100%|██████████| 17613/17613 [00:01<00:00, 10326.13it/s]


train : 7704 images
valid : 550 images
test  : 550 images
nc = 27
✅ Dataset prepared!


In [3]:
%%writefile dsppf_backbone.py
"""Paper 6 backbone for YOLO26n: SPPF at layer 9 -> DS-SPPF (Dynamic Scale SPPF)."""
import torch
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import SPPF
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

DSSPPF_LAYER = 9
N_BRANCH = 4


class DSSPPF(nn.Module):
    """Dynamic Scale SPPF: identity, MaxPool and two dilated depthwise branches, fused with per-pixel, per-channel softmax weights."""

    def __init__(self, c1, c2, k=5, shortcut=False):
        super().__init__()
        c_ = c1 // 2  # hidden channels (same as SPPF)
        self.cv1 = Conv(c1, c_, 1, 1, act=False)
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)
        self.dw2 = Conv(c_, c_, 3, 1, g=c_, d=2)
        self.dw3 = Conv(c_, c_, 3, 1, g=c_, d=3)
        self.gen = nn.Sequential(nn.Conv2d(c_, c_, 3, 1, 1), nn.SiLU(), nn.Conv2d(c_, N_BRANCH * c_, 1))
        nn.init.zeros_(self.gen[-1].weight)  # equal branch weights (0.25) at the start
        nn.init.zeros_(self.gen[-1].bias)
        self.cv2 = Conv(N_BRANCH * c_, c2, 1, 1)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        f = self.cv1(x)
        branches = (f, self.m(f), self.dw2(f), self.dw3(f))
        # softmax across the 4 branches, written with 4-D ops only (NCNN friendly)
        logits = self.gen(f).chunk(N_BRANCH, 1)
        mx = torch.maximum(torch.maximum(logits[0], logits[1]), torch.maximum(logits[2], logits[3]))
        e = [torch.exp(l - mx) for l in logits]
        s = e[0] + e[1] + e[2] + e[3]
        y = self.cv2(torch.cat([b * w / s for b, w in zip(branches, e)], 1))
        return y + x if self.add else y


def add_dsppf(model, layer=DSSPPF_LAYER):
    """Replace the SPPF at the given layer with DS-SPPF, copying cv1/cv2 from the SPPF. Returns the model."""
    old = model.model[layer]
    assert isinstance(old, SPPF), f"layer {layer} is {type(old).__name__}, not SPPF"
    c1, c_, c2 = old.cv1.conv.in_channels, old.cv1.conv.out_channels, old.cv2.conv.out_channels
    assert old.cv2.conv.in_channels == N_BRANCH * c_, "SPPF concat width does not match the 4 DS-SPPF branches"
    new = DSSPPF(c1, c2, k=old.m.kernel_size)
    new.add = old.add  # same shortcut as the base SPPF
    new.cv1.load_state_dict(old.cv1.state_dict())
    new.cv2.load_state_dict(old.cv2.state_dict())
    with torch.no_grad():
        new.cv2.conv.weight.mul_(N_BRANCH)  # cancels the initial 0.25 branch weights
    new = new.to(next(old.parameters()).device)
    new.i, new.f, new.type = old.i, old.f, "DSSPPF"  # keep the routing attributes used by the model's forward
    new.np = sum(p.numel() for p in new.parameters())
    model.model[layer] = new
    return model


class DSSPPFTrainer(DetectionTrainer):
    """DetectionTrainer that loads the pretrained weights first, then swaps SPPF for DS-SPPF (cv1/cv2 keep the pretrained weights)."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_dsppf(model)
        if RANK in {-1, 0}:
            print(f"Paper 6 backbone: DS-SPPF at layer {DSSPPF_LAYER}")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


Writing dsppf_backbone.py


In [4]:
import importlib, dsppf_backbone
importlib.reload(dsppf_backbone)
from dsppf_backbone import add_dsppf, DSSPPFTrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs Paper 6 backbone
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
dsppf = add_dsppf(deepcopy(base)).eval()
x = torch.rand(1, 3, 640, 640)
with torch.no_grad():
    feat = x
    for m in base.model[:9]:  # backbone layers 0-8 are sequential (from = -1)
        assert m.f == -1
        feat = m(feat)
    # layer 9 output at initialisation: cv1/cv2 copied, branch weights 0.25, cv2 x4
    out_base, out_ds = base.model[9](feat), dsppf.model[9](feat)
print("layer 9:", type(dsppf.model[9]).__name__, "| shortcut:", dsppf.model[9].add, "| output", tuple(out_ds.shape))
print(f"layer 9 output, mean |base - DS-SPPF| at init: {(out_base - out_ds).abs().mean():.4f} (not 0: the depthwise branches are new)")
_ = dsppf(torch.zeros(1, 3, 640, 640))  # full forward check
for tag, m in (("base", base), ("Paper 6 backbone", dsppf)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:17s} Params {pm:.3f} M | GFLOPs {gf:.2f}")

Overriding model.yaml nc=80 with nc=27
layer 9: DSSPPF | shortcut: True | output (1, 256, 20, 20)
layer 9 output, mean |base - DS-SPPF| at init: 0.0000 (not 0: the depthwise branches are new)
base              Params 2.380 M | GFLOPs 5.34
Paper 6 backbone  Params 2.596 M | GFLOPs 5.51


In [5]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("DS-SPPF Backbone (Paper 6)", "bb_dsppf"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=DSSPPFTrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()


========== DS-SPPF Backbone (Paper 6) (bb_dsppf) ==========
New https://pypi.org/project/ultralytics/8.4.183 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/digitalaccess/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=No

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      3.25G      1.361      2.295    0.01843         14        640: 100% ━━━━━━━━━━━━ 482/482 4.6it/s 1:44
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.0it/s 2.6s
                   all        550        557      0.795      0.719      0.868      0.501

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      3.26G      1.402      1.562    0.01904         12        640: 100% ━━━━━━━━━━━━ 482/482 4.8it/s 1:40
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 7.1it/s 2.5s
                   all        550        557      0.786      0.781      0.897      0.537

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100      3.26G      1.421      1.278    0.01932          7        640: 100% ━━━━━━━━━━━━ 482/482 4.9it/s 1:39
                 Class     Images  Instances      Box(P 

In [6]:
import dsppf_backbone  # must be imported before loading a DS-SPPF checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 127 layers, 2,596,293 parameters, 0 gradients, 5.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 973.4±262.2 MB/s, size: 37.7 KB)
val: Scanning /kaggle/working/digitalaccess/test/labels... 550 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 550/550 1.1Kit/s 0.5s
val: New cache created: /kaggle/working/digitalaccess/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 35/35 12.9it/s 2.7s
                   all        550        552      0.972      0.943      0.969      0.589
Speed: 0.6ms preprocess, 2.5ms inference, 0.0ms loss, 0.1ms postprocess per image


,Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50
0,DS-SPPF Backbone (Paper 6),2.6,5.51,0.972,0.943,0.957,0.969


In [7]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper6_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("dsppf_backbone.py", EDGE / "dsppf_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import dsppf_backbone  # needed to load the DS-SPPF checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper6_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper6_backbone_edge_bench.zip from the Output tab.")

bench_images: 50 real test images
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.137 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ This export format does not support end2end models, disabling the end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 151 layers, 2,721,026 parameters, 0 gradients, 5.5 GFLOPs

PyTorch: starting from '/kaggle/working/runs/bb_dsppf_s0/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 31, 8400) (5.6 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 1 package in 251ms
 Downloaded ncnn
Prepared 1 package in 217ms
Installed 1 package in 3ms
 + ncnn==1.0.20261020

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: R

pnnxparam = /kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model/model.pnnx.param
pnnxbin = /kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model/model.pnnx.bin
pnnxpy = /kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model/model_pnnx.py
pnnxonnx = /kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model/model.pnnx.onnx
ncnnparam = /kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model/model.ncnn.param
ncnnbin = /kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model/model.ncnn.bin
ncnnpy = /kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model/model_ncnn.py
fp16 = 0
optlevel = 2
device = cpu
inputshape = [1,3,640,640]f32
inputshape2 = 
input = 
input2 = 
customop = 
moduleop = 
get inputshape from traced inputs
inputshape = [1,3,640,640]f32
############# pass_level0
inline module = dsppf_backbone.DSSPPF
inline module = torch.nn.modules.linear.Identity
inline module = ultralytics.nn.modules.block.Attention
inline module = ultralytics.nn.modules.block.Bottleneck
inline mo

NCNN: export success ✅ 7.8s, saved as '/kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model' (10.1 MB)

Export complete (8.4s)
Results saved to /kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/runs/bb_dsppf_s0/weights/best_ncnn_model imgsz=640 data=/kaggle/working/digitalaccess/data.yaml  
Visualize:       https://netron.app
bb_dsppf     -> bb_dsppf.pt, bb_dsppf_ncnn_model/

Download paper6_backbone_edge_bench.zip from the Output tab.


In [8]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_dsppf": None}
FPS_JETSON = {"bb_dsppf": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


Backbone Variant,Params (M),GFLOPs,Precision,Recall,F1-score,mAP@50,FPS (Rasp-5),FPS (Jetson)
Standard YOLO26n Backbone,2.380000,5.340000,0.971000,0.937000,0.953000,0.958000,13.380000,--
DS-SPPF Backbone (Paper 6),2.600000,5.510000,0.972000,0.943000,0.957000,0.969000,--,--


In [9]:
import zipfile

ALL_ZIP = WORK / "paper6_backbone_all_results.zip"
folders = [WORK / "paper6_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")

paper6_backbone_edge_bench/    59 files
runs/                          26 files
weights/bb_dsppf_best.pt

Download paper6_backbone_all_results.zip (47.0 MB) from the Output tab.
